In [1]:
from hydra import initialize, compose
import sys
import pandas as pd
from pathlib import Path

# sys.path.append("/home/psa_images/SemiF-AnnotationPipeline/repo_overview")
from repo_overview import ParquetDataProcessor, SampleImageData
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import seaborn as sns
import os
import matplotlib.pyplot as plt
import cv2
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
import random
import hydra
from repo_overview_utils import classify_area

from omegaconf import OmegaConf

from hydra.core.hydra_config import HydraConfig

## Helpers

In [ ]:
def get_image_info(row, idx):
    cutpath = row["cutout_path"]
    common_name = row["common_name"]
    genus = row["genus"]
    species = row["species"]

    info_dict = {
        "index": idx,
        "common_name": common_name,
        "genus": genus,
        "species": species,
        "cutout_path": cutpath,
    }
    return info_dict


def filter_area(df, lower, upper):
    filtered_dfs = []
    for spec in df.common_name.unique():
        temp = df[df["common_name"] == spec]

        mean = temp.area.describe()["mean"]
        min = temp.area.describe()["min"]
        max = temp.area.describe()["max"]
        per25 = temp.area.describe()["25%"]
        per50 = temp.area.describe()["50%"]
        per75 = temp.area.describe()["75%"]

        if type(lower) is int:
            lower_area_limit = lower
        if lower is None:
            lower_area_limit = 0
        if lower == "mean":
            lower_area_limit = mean
        if lower == "min":
            lower_area_limit = min
        if lower == "max":
            lower_area_limit = max
        if lower == "per25":
            lower_area_limit = per25
        if lower == "per50":
            lower_area_limit = per50
        if lower == "per75":
            lower_area_limit = per75

        if type(upper) is int:
            upper_area_limit = upper
        if upper == "mean":
            upper_area_limit = mean
        if upper == "min":
            upper_area_limit = min
        if upper == "max":
            upper_area_limit = max
        if upper == "per25":
            upper_area_limit = per25
        if upper == "per50":
            upper_area_limit = per50
        if upper == "per75":
            upper_area_limit = per75

        temp = temp[
            (temp["area"] < upper_area_limit) & (lower_area_limit < temp["area"])
        ]
        filtered_dfs.append(temp)
    filtered_df = pd.concat(filtered_dfs)
    return filtered_df


def filter_green_sum(df, lower, upper):
    filtered_dfs = []
    for spec in df.common_name.unique():
        temp = df[df["common_name"] == spec]

        mean = temp.green_sum.describe()["mean"]
        min = temp.green_sum.describe()["min"]
        max = temp.green_sum.describe()["max"]
        per25 = temp.green_sum.describe()["25%"]
        per50 = temp.green_sum.describe()["50%"]
        per75 = temp.green_sum.describe()["75%"]
        if type(lower) is int:
            lower_green_sum_limit = lower
        if lower is None:
            lower_green_sum_limit = 0
        if lower == "mean":
            lower_green_sum_limit = mean
        if lower == "min":
            lower_green_sum_limit = min
        if lower == "max":
            lower_green_sum_limit = max
        if lower == "per25":
            lower_green_sum_limit = per25
        if lower == "per50":
            lower_green_sum_limit = per50
        if lower == "per75":
            lower_green_sum_limit = per75

        if type(upper) is int:
            upper_green_sum_limit = upper
        if upper == "mean":
            upper_green_sum_limit = mean
        if upper == "min":
            upper_green_sum_limit = min
        if upper == "max":
            upper_green_sum_limit = max
        if upper == "per25":
            upper_green_sum_limit = per25
        if upper == "per50":
            upper_green_sum_limit = per50
        if upper == "per75":
            upper_green_sum_limit = per75

        temp = temp[
            (temp["green_sum"] < upper_green_sum_limit)
            & (lower_green_sum_limit < temp["green_sum"])
        ]
        filtered_dfs.append(temp)
    filtered_df = pd.concat(filtered_dfs)
    return filtered_df

## Load config

In [ ]:
with initialize(version_base="1.3", config_path="../conf"):
    cfg = compose(config_name="config.yaml", return_hydra_config=True)
    cfg.general.batch_id = "repo_overview"
    cfg.hydra.runtime.cwd = "/home/psa_images/SemiF-AnnotationPipeline"
    cfg.data.database_parquet = (
        "/home/psa_images/SemiF-AnnotationPipeline/repo_overview/database"
    )
    cfg.data.repo_database = "/home/psa_images/SemiF-AnnotationPipeline/repo_overview"

## Read database into a dataframe

In [ ]:
processor = ParquetDataProcessor(cfg)
processor.read_parquet_file(totals=False)

## Minor cleaning

In [ ]:
df = processor.df.copy()
cols = [
    "EPPO",
    "USDA_symbol",
    "area",
    "authority",
    "axis_major_length",
    "axis_minor_length",
    "b",
    "b_mean",
    "b_std",
    "batch_id",
    "bbox",
    "blob_home",
    "blur_effect",
    "camera_info",
    "category",
    "class",
    "class_id",
    "collection_location",
    "collection_timing",
    "common_name",
    "cropout_b_mean",
    "cropout_b_std",
    "cropout_exg_mean",
    "cropout_exg_std",
    "cropout_g_mean",
    "cropout_g_std",
    "cropout_r_mean",
    "cropout_r_std",
    "cutout_b_mean",
    "cutout_b_std",
    "cutout_exg_mean",
    "cutout_exg_std",
    "cutout_g_mean",
    "cutout_g_std",
    "cutout_id",
    "cutout_num",
    "cutout_path",
    "cutout_r_mean",
    "cutout_r_std",
    "cutout_version",
    "data_root",
    "date",
    "datetime",
    "dt",
    "duration",
    "eccentricity",
    "exg_sum",
    "exif_meta",
    "extends_border",
    "extent",
    "family",
    "g",
    "g_max",
    "g_mean",
    "g_min",
    "g_std",
    "general_season",
    "genus",
    "green_sum",
    "group",
    "growth_habit",
    "hex",
    "hwc",
    "image_id",
    "is_primary",
    "link",
    "multi_species_USDA_symbol",
    "note",
    "num_components",
    "order",
    "perimeter",
    "r",
    "r_max",
    "r_mean",
    "r_min",
    "r_std",
    "schema_version",
    "season",
    "shape",
    "solidity",
    "species",
    "state_id",
    "subclass",
    "synth",
]
df = df[cols]
df = df[df["common_name"] != "Unknown"]
df = df[df["common_name"] != "Colorchecker"]
df["file_path"] = cfg.data.longterm_storage + "/semifield-cutouts/" + df["cutout_path"]

In [ ]:
df.groupby(["general_season", "common_name"])[
    "image_id"
].nunique().reset_index().sort_values(by=["general_season"])
df.groupby(["general_season", "common_name"])[
    "cutout_id"
].nunique().reset_index().sort_values(by=["general_season"]).sum()

### Creating an "area_class" column

In [ ]:
# Apply the function to create the 'area_class' column
df["area_class"] = df["area"].apply(classify_area)

## Filtering

In [ ]:
df["common_name"].unique()

In [ ]:
filtereddf = df[df["general_season"] == "cover crops"]
filtereddf = filtereddf[filtereddf["common_name"] != "Soybean"]
filtereddf = filtereddf[filtereddf["common_name"] != "Upland cotton"]
filtereddf = filtereddf[filtereddf["common_name"] == "Cereal rye"]

# filtereddf = filtereddf.drop_duplicates("cutout_id")
filtereddf = filtereddf[filtereddf["is_primary"] == True]
filtereddf = filtereddf[filtereddf["extends_border"] == False].reset_index(drop=True)

lower_area_limit = 100000
upper_area_limit = 10000000000
filtereddf = filter_area(filtereddf, "per75", "max")
filtereddf.shape

## Cutout Plotting

### Inspect and save a sample of the results

In [ ]:
plt.close("all")
plt.style.context("ggplot")
randint = random.randint(0, 1000)
# randint = 947
print(randint)
savecsv = False
savefig = False
sampled_df = filtereddf.sample(n=16, random_state=randint)

# Create a 4x4 grid of plots
# fig, axs = plt.subplots(4, 4, figsize=(20, 20))

# Flatten the array of axes for easy iteration
# axs = axs.ravel()
path_list = []
for i, (index, row) in enumerate(sampled_df.iterrows()):
    # Check if the file exists to avoid errors
    cropoutpath = row["file_path"].replace(".png", ".jpg")
    cutoutpath = row["file_path"]
    if os.path.exists(cutoutpath):
        # Load the image
        # path = row["file_path"]
        cutpath = row["cutout_path"]
        common_name = row["common_name"]
        genus = row["genus"]
        species = row["species"]

        cropout = cv2.cvtColor(cv2.imread(cropoutpath), cv2.COLOR_BGR2RGB)
        cutout = cv2.cvtColor(cv2.imread(cutoutpath), cv2.COLOR_BGR2RGB)

        # img = mpimg.imread(path)
        path_list.append(
            {
                "index": index,
                "common_name": common_name,
                "genus": genus,
                "species": species,
                "cutout_path": cutpath,
                "seed_number": randint,
            }
        )
        fig, axs = plt.subplots(1, 2, figsize=(12, 8))

        axs[0].imshow(cropout)
        axs[0].axis("off")
        axs[0].set_title(f"{cutout.shape}")
        axs[1].imshow(cutout)
        axs[1].axis("off")

        # Display the image
        # axs[i].imshow(img)
        # axs[i].axis("off")  # Hide axes ticks

        fontsize = 16
        formatted_title = f"{index} | {common_name} (${genus.title()}$ ${species}$)"
        axs[1].set_title(formatted_title, fontsize=fontsize)
        # axs.title(formatted_title, fontsize=fontsize)
        plt.show()
        # axs[i].set_title(formatted_title, fontsize=fontsize)
    else:
        continue
        # plt.set_visible(False)
    # axs[i].set_visible(
    #     False
    # )  # Hide the subplot if the file doesn't exist or can't be opened

# plt.tight_layout()
plt.show()
    
plt.close("all")
if savecsv:
    tdf = pd.DataFrame(path_list)
    tdf.to_csv("figures/imgs_list_for_figure.csv", mode="a", index=False)

### Check masks

In [ ]:
# Create a 4x4 grid of plots
fig, axs = plt.subplots(4, 4, figsize=(20, 20))

# Flatten the array of axes for easy iteration
axs = axs.ravel()
path_list = []
for i, (index, row) in enumerate(sampled_df.iterrows()):
    # Check if the file exists to avoid errors
    path = row["file_path"]  # .replace(".png", ".jpg")
    if os.path.exists(path):
        # Load the image
        # path = row["file_path"]
        cutpath = row["cutout_path"]
        common_name = row["common_name"]
        genus = row["genus"]
        species = row["species"]

        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        print(img.shape)
        # img = mpimg.imread(path)
        path_list.append(
            {
                "index": index,
                "common_name": common_name,
                "genus": genus,
                "species": species,
                "cutout_path": cutpath,
                "seed_number": randint,
            }
        )
        # Display the image
        axs[i].imshow(img)
        axs[i].axis("off")  # Hide axes ticks
        formatted_title = f"{index} | {common_name} (${genus.title()}$ ${species}$)"
        # plt.title(f"$\it{text}$ ~ {text2}")
        fontsize = 16
        axs[i].set_title(formatted_title, fontsize=fontsize)
    else:
        axs[i].set_visible(
            False
        )  # Hide the subplot if the file doesn't exist or can't be opened

plt.tight_layout()
plt.show()

## Plot using saved list

In [ ]:
import shutil

figdf = pd.read_csv("imgs_list_for_figure_cashcrops.csv").sort_values(by="genus")
for i, (index, row) in enumerate(figdf.iterrows()):
    cutoutpath = cfg.data.longterm_storage + "/semifield-cutouts/" + row["cutout_path"]
    cropoutpath = cutoutpath.replace(".png", ".jpg")
    # shutil.copy2(cutoutpath, "cashcrops")
    # shutil.copy2(cropoutpath, "cashcrops")

In [ ]:
figdf = figdf.iloc[:3]
figdf

## Images of Weed Cropouts

In [ ]:
plt.style.context("ggplot")
fig_df = pd.read_csv("figures/imgs_list_for_figure.csv").sort_values(by="genus")
# Create a 4x4 grid of plots
fig, axs = plt.subplots(4, 4, figsize=(20, 20))
# Flatten the array of axes for easy iterationa
axs = axs.ravel()
path_list = []
for i, (index, row) in enumerate(fig_df.iterrows()):
    # Check if the file exists to avoid errors
    cutoutpath = Path(
        cfg.data.longterm_storage,
        "semifield-cutouts",
        row["cutout_path"].replace(".png", ".jpg"),
    )

    if cutoutpath.exists():
        common_name = row["common_name"]
        genus = row["genus"]
        species = row["species"]

        img = cv2.cvtColor(cv2.imread(str(cutoutpath)), cv2.COLOR_BGR2RGB)
        # img = mpimg.imread(cutoutpath)
        axs[i].imshow(img)
        axs[i].axis("off")  # Hide axes ticks
        formatted_title = f"{common_name} (${genus.title()}$ ${species}$)"
        # plt.title(f"$\it{text}$ ~ {text2}")
        fontsize = 14
        axs[i].set_title(formatted_title, fontsize=fontsize)
    else:
        axs[i].set_visible(
            False
        )  # Hide the subplot if the file doesn't exist or can't be opened

plt.tight_layout()
transparency = True

plt.savefig(
    f"figures/Weeds_cropout_plot_transparency_{transparency}_fontsize{fontsize}.png",
    dpi=300,
    bbox_inches="tight",
    transparent=transparency,
    pad_inches=0,
)
plt.show()

##  Cash crops

In [ ]:
plt.style.context("ggplot")
fig_df = pd.read_csv("figures/imgs_list_for_figure_cashcrops.csv").sort_values(
    by="genus"
)
# Create a 4x4 grid of plots
fig, axs = plt.subplots(4, 4, figsize=(20, 20))
# Flatten the array of axes for easy iterationa
axs = axs.ravel()
path_list = []
for i, (index, row) in enumerate(fig_df.iterrows()):
    # Check if the file exists to avoid errors
    cutoutpath = Path(
        cfg.data.longterm_storage,
        "semifield-cutouts",
        row["cutout_path"].replace(".png", ".jpg"),
    )

    if cutoutpath.exists():
        common_name = row["common_name"]
        genus = row["genus"]
        species = row["species"]

        img = cv2.cvtColor(cv2.imread(str(cutoutpath)), cv2.COLOR_BGR2RGB)
        # img = mpimg.imread(cutoutpath)
        axs[i].imshow(img)
        axs[i].axis("off")  # Hide axes ticks
        formatted_title = f"{common_name} (${genus.title()}$ ${species}$)"
        # plt.title(f"$\it{text}$ ~ {text2}")
        fontsize = 14
        axs[i].set_title(formatted_title, fontsize=fontsize)
    else:
        axs[i].set_visible(
            False
        )  # Hide the subplot if the file doesn't exist or can't be opened

plt.tight_layout()
transparency = True
save = True
if save:
    plt.savefig(
        f"figures/Weeds_cropout_plot_transparency_{transparency}_fontsize{fontsize}.png",
        dpi=300,
        bbox_inches="tight",
        transparent=transparency,
        pad_inches=0,
    )
plt.show()

## Non-target weeds

In [ ]:
def get_image_info(row, idx):
    cutpath = row["cutout_path"]
    common_name = row["common_name"]
    genus = row["genus"]
    species = row["species"]

    info_dict = {
        "index": idx,
        "common_name": common_name,
        "genus": genus,
        "species": species,
        "cutout_path": cutpath,
    }
    return info_dict


def filter_area(df, lower, upper):
    filtered_dfs = []
    for spec in df.common_name.unique():
        temp = df[df["common_name"] == spec]

        mean = temp.area.describe()["mean"]
        min = temp.area.describe()["min"]
        max = temp.area.describe()["max"]
        per25 = temp.area.describe()["25%"]
        per50 = temp.area.describe()["50%"]
        per75 = temp.area.describe()["75%"]

        if type(lower) is int:
            lower_area_limit = lower
        if lower is None:
            lower_area_limit = 0
        if lower == "mean":
            lower_area_limit = mean
        if lower == "min":
            lower_area_limit = min
        if lower == "max":
            lower_area_limit = max
        if lower == "per25":
            lower_area_limit = per25
        if lower == "per50":
            lower_area_limit = per50
        if lower == "per75":
            lower_area_limit = per75

        if type(upper) is int:
            upper_area_limit = upper
        if upper == "mean":
            upper_area_limit = mean
        if upper == "min":
            upper_area_limit = min
        if upper == "max":
            upper_area_limit = max
        if upper == "per25":
            upper_area_limit = per25
        if upper == "per50":
            upper_area_limit = per50
        if upper == "per75":
            upper_area_limit = per75

        temp = temp[
            (temp["area"] < upper_area_limit) & (lower_area_limit < temp["area"])
        ]
        filtered_dfs.append(temp)
    filtered_df = pd.concat(filtered_dfs)
    return filtered_df


def filter_green_sum(df, lower, upper):
    filtered_dfs = []
    for spec in df.common_name.unique():
        temp = df[df["common_name"] == spec]

        mean = temp.green_sum.describe()["mean"]
        min = temp.green_sum.describe()["min"]
        max = temp.green_sum.describe()["max"]
        per25 = temp.green_sum.describe()["25%"]
        per50 = temp.green_sum.describe()["50%"]
        per75 = temp.green_sum.describe()["75%"]
        if type(lower) is int:
            lower_green_sum_limit = lower
        if lower is None:
            lower_green_sum_limit = 0
        if lower == "mean":
            lower_green_sum_limit = mean
        if lower == "min":
            lower_green_sum_limit = min
        if lower == "max":
            lower_green_sum_limit = max
        if lower == "per25":
            lower_green_sum_limit = per25
        if lower == "per50":
            lower_green_sum_limit = per50
        if lower == "per75":
            lower_green_sum_limit = per75

        if type(upper) is int:
            upper_green_sum_limit = upper
        if upper == "mean":
            upper_green_sum_limit = mean
        if upper == "min":
            upper_green_sum_limit = min
        if upper == "max":
            upper_green_sum_limit = max
        if upper == "per25":
            upper_green_sum_limit = per25
        if upper == "per50":
            upper_green_sum_limit = per50
        if upper == "per75":
            upper_green_sum_limit = per75

        temp = temp[
            (temp["green_sum"] < upper_green_sum_limit)
            & (lower_green_sum_limit < temp["green_sum"])
        ]
        filtered_dfs.append(temp)
    filtered_df = pd.concat(filtered_dfs)
    return filtered_df

### Filtering

In [ ]:
filtereddf = df[df["general_season"] == "weeds"]
filtereddf = filtereddf[filtereddf["common_name"] != "Soybean"]
filtereddf = filtereddf[filtereddf["common_name"] != "Upland cotton"]
filtereddf = filtereddf[filtereddf["common_name"] != "Fall panicum"]
filtereddf = filtereddf[filtereddf["common_name"] != "Yellow foxtail"]
filtereddf = filtereddf[filtereddf["common_name"] != "Giant foxtail"]
filtereddf = filtereddf[filtereddf["common_name"] != "Sprawling signalgrass"]
# filtereddf = filtereddf[filtereddf["state_id"] != "TX"]
filtereddf = filtereddf[filtereddf["state_id"] == "NC"]

filtereddf.shape

In [ ]:
filtered_df = filter_area(filtereddf, 0, "per25")
filtered_df = filter_green_sum(filtered_df, 0, "per25")

filtered_df.shape

In [ ]:
plt.style.context("ggplot")
randint = random.randint(0, 1000)
randint = 873
savecsv = True
savefig = False

print(randint)
sampled_df = filtered_df.sample(n=16, random_state=randint, replace=True)
# Create a 4x4 grid of plots
fig, axs = plt.subplots(4, 4, figsize=(12, 8))
# Flatten the array of axes for easy iterationa
axs = axs.ravel()
info = []
for i, (index, row) in enumerate(sampled_df.iterrows()):
    # Check if the file exists to avoid errors
    cutoutpath = Path(
        cfg.data.longterm_storage,
        "semifield-cutouts",
        row["cutout_path"].replace(".png", ".jpg"),
    )

    if cutoutpath.exists():
        info_dict = get_image_info(row, i)
        info.append(info_dict)

        img = cv2.cvtColor(cv2.imread(str(cutoutpath)), cv2.COLOR_BGR2RGB)
        # img = mpimg.imread(cutoutpath)
        axs[i].imshow(img)
        axs[i].axis("off")  # Hide axes ticks
        formatted_title = f"{info_dict['index']} | {info_dict['common_name']}"  # (${info_dict['genus'].title()}$ ${info_dict['species']}$)"
        # plt.title(f"$\it{text}$ ~ {text2}")
        fontsize = 14
        axs[i].set_title(formatted_title, fontsize=fontsize)
    else:
        axs[i].set_visible(
            False
        )  # Hide the subplot if the file doesn't exist or can't be opened

    plt.tight_layout()
transparency = True

if savefig:
    plt.savefig(
        f"figures/Weeds_cropout_plot_transparency_{transparency}_fontsize{fontsize}.png",
        dpi=300,
        bbox_inches="tight",
        transparent=transparency,
        pad_inches=0,
    )
if savecsv:
    tdf = pd.DataFrame(info)
    tdf.to_csv("figures/imgs_list_for_non-target_weeds.csv", mode="a", index=False)
plt.show()

In [ ]:
plt.style.context("ggplot")
fig_df = pd.read_csv(
    "figures/imgs_list_for_non-target_weeds.csv"
)  # .sort_values(by="genus")
print(fig_df.shape)
# Create a 4x4 grid of plots
fig, axs = plt.subplots(3, 3, figsize=(20, 20))
# Flatten the array of axes for easy iterationa
axs = axs.ravel()
path_list = []
for i, (index, row) in enumerate(fig_df.iterrows()):
    # Check if the file exists to avoid errors
    cutoutpath = Path(
        cfg.data.longterm_storage,
        "semifield-cutouts",
        row["cutout_path"].replace(".png", ".jpg"),
    )

    if cutoutpath.exists():
        common_name = row["common_name"]
        genus = row["genus"]
        species = row["species"]
        index = row["index"]

        img = cv2.cvtColor(cv2.imread(str(cutoutpath)), cv2.COLOR_BGR2RGB)
        # img = mpimg.imread(cutoutpath)
        axs[i].imshow(img)
        axs[i].axis("off")  # Hide axes ticks
        formatted_title = f"{index} | {common_name} (${genus.title()}$ ${species}$)"
        # plt.title(f"$\it{text}$ ~ {text2}")
        fontsize = 14
        # axs[i].set_title(formatted_title, fontsize=fontsize)
    else:
        axs[i].set_visible(
            False
        )  # Hide the subplot if the file doesn't exist or can't be opened
# Letters to overlay
letters = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]

# Loop through each subplot and overlay a letter
for ax, letter in zip(axs.flat, letters):
    ax.text(
        0.05,
        0.95,
        letter,
        transform=ax.transAxes,
        fontsize=48,
        color="white",
        ha="left",
        va="top",
        weight="bold",
    )

# Adjust layout to prevent overlap
plt.tight_layout()
plt.tight_layout()
transparency = True
savefig = True
if savefig:
    plt.savefig(
        f"figures/non-target_weeds_transparency_{transparency}_fontsize{fontsize}.png",
        dpi=300,
        bbox_inches="tight",
        transparent=transparency,
        pad_inches=0,
    )
plt.show()

## Inset

In [ ]:
import numpy
import matplotlib.pyplot as plt

plt.style.context("ggplot")
fig_df = pd.read_csv("figures/imgs_list_for_figure.csv").sort_values(by="genus")
# Create a 4x4 grid of plots
fig, axs = plt.subplots(4, 4, figsize=(20, 20))
# Flatten the array of axes for easy iterationa
axs = axs.ravel()
path_list = []
for i, (index, row) in enumerate(fig_df.iterrows()):

    cutoutpath = Path(
        cfg.data.longterm_storage, "semifield-cutouts", row["cutout_path"]
    )
    cropoutpath = Path(
        cfg.data.longterm_storage,
        "semifield-cutouts",
        row["cutout_path"].replace(".png", ".jpg"),
    )

    cutout = cv2.cvtColor(cv2.imread(str(cutoutpath)), cv2.COLOR_BGR2RGB)
    cropout = cv2.cvtColor(cv2.imread(str(cropoutpath)), cv2.COLOR_BGR2RGB)
    # fig = plt.figure(figsize=(18, 12))
    # ax1 = fig.add_subplot(2,1,1)
    axs[i].imshow(cropout)

    inset2 = fig.add_axes(
        (0.525, 0.535, 0.1, 0.1), sharex=axs[i], sharey=axs[i]
    )  # tuple (left, bottom, width, height)
    inset2.imshow(cutout)
    plt.setp(inset2, xticks=[], yticks=[])
    formatted_title = f"{common_name} (${genus.title()}$ ${species}$)"
    axs[i].set_title(formatted_title, fontsize=fontsize)
plt.show()

#     # Check if the file exists to avoid errors
#     cutoutpath = Path(cfg.data.longterm_storage, "semifield-cutouts",row["cutout_path"].replace(".png", ".jpg"))

#     if cutoutpath.exists():
#         common_name = row["common_name"]
#         genus = row["genus"]
#         species = row["species"]

#         img = cv2.cvtColor(cv2.imread(str(cutoutpath)), cv2.COLOR_BGR2RGB)
#         # img = mpimg.imread(cutoutpath)
#         axs[i].imshow(img)
#         axs[i].axis("off")  # Hide axes ticks
#         formatted_title = f"{common_name} (${genus.title()}$ ${species}$)"
#         # plt.title(f"$\it{text}$ ~ {text2}")
#         fontsize = 14
#         axs[i].set_title(formatted_title, fontsize=fontsize)
#     else:
#         axs[i].set_visible(
#             False
#         )  # Hide the subplot if the file doesn't exist or can't be opened

# plt.tight_layout()

In [ ]:
fig_df = pd.read_csv("figures/imgs_list_for_figure.csv").sort_values(by="genus")
row = fig_df.iloc[15]


cutoutpath = Path(cfg.data.longterm_storage, "semifield-cutouts", row["cutout_path"])
cropoutpath = Path(
    cfg.data.longterm_storage,
    "semifield-cutouts",
    row["cutout_path"].replace(".png", ".jpg"),
)

cutout = cv2.cvtColor(cv2.imread(str(cutoutpath)), cv2.COLOR_BGR2RGB)
cropout = cv2.cvtColor(cv2.imread(str(cropoutpath)), cv2.COLOR_BGR2RGB)
fig = plt.figure(figsize=(18, 12))
ax1 = fig.add_subplot(1, 1, 1)
ax1.imshow(cropout)

formatted_title = f"{common_name} (${genus.title()}$ ${species}$)"
# ax1.set_title(formatted_title, fontsize=fontsize)

inset2 = fig.add_axes(
    (0.6, 0.12, 0.2, 0.2), sharex=ax1, sharey=ax1
)  # tuple (left, bottom, width, height)
inset2.imshow(cutout)
plt.setp(inset2, xticks=[], yticks=[])

transparency = True
save = False
if save:
    name = row["common_name"]
    plt.savefig(
        f"figures/inset/{name}_inset.png",
        dpi=300,
        bbox_inches="tight",
        transparent=transparency,
        pad_inches=0,
    )
plt.show()

In [ ]:
# figdf = pd.read_csv("imgs_list_for_figure_cashcrops.csv")  # .sort_values(by="genus")
figdf = pd.read_csv(
    "figures/imgs_list_for_figure_weesd.csv"
)  # .sort_values(by="genus")
figdf = figdf.iloc[:3]
# Create a 4x4 grid of plots
fig, axs = plt.subplots(4, 4, figsize=(10, 20))  # Adjusted for pairs of images
# Flatten the array of axes for easy iteration
axs = axs.ravel()
path_list = []
for i, row in figdf.iterrows():
    mask_image_path = (
        cfg.data.longterm_storage + "/semifield-cutouts/" + row["cutout_path"]
    )
    color_image_path = mask_image_path.replace(".png", ".jpg")
    # Check if both files exist to avoid errors
    if os.path.exists(color_image_path) and os.path.exists(mask_image_path):
        # Load the color image and mask
        color_img = mpimg.imread(color_image_path)
        mask_img = mpimg.imread(mask_image_path)

        # Display the color image
        axs[2 * i].imshow(color_img)
        axs[2 * i].axis("off")  # Hide axes ticks
        fontsize = 18
        axs[2 * i].set_title(
            f"{row['common_name']} ({row['genus']} {row['species']})", fontsize=fontsize
        )

        # Display the mask image
        axs[2 * i + 1].imshow(mask_img)
        axs[2 * i + 1].axis("off")  # Hide axes ticks

    else:
        # Hide the subplot if the files don't exist or can't be opened
        axs[2 * i].set_visible(False)
        axs[2 * i + 1].set_visible(False)


plt.tight_layout()
plt.subplots_adjust(wspace=0, hspace=-0.15)
transparency = False
# plt.savefig(
#     f"figures/Cash_crops_Cropout_plot_transparency_{transparency}_fontsize{fontsize}.png",
#     dpi=300,
#     bbox_inches="tight",
#     transparent=transparency,
#     pad_inches=0,
# )
plt.show()